In [3]:
# coding=utf-8
# Copyright (c) 2025 Huawei Technologies Co.,
# All rights reserved.
#
# Example inference script for fine-tuned openPangu-Embedded-1B on NPU.
#
# Usage:
#   python generate.py \
#       --model_path /opt/huawei/edu-apaas/src/init/luoyuping/pangu1B_training/outputs/float_test/bf16_false/final_model \
#       --prompt "Explain the concept of neural networks in simple terms."
#
# When you push model to Hugging Face, you can set --model_path to your repo id,
# e.g. --model_path your-username/openPangu-Embedded-1B-finetuned

import argparse
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig


def load_model_and_tokenizer(model_path: str):
    """
    Load tokenizer and causal LM model from local path or HF hub.
    """

    # 说明：
    # - trust_remote_code=True 是因为 openPangu-Embedded-1B 通常带有自定义模型实现
    # - local_files_only=True：当前场景使用本地微调后模型。如果你上传到 HF 后想直接从 Hub 拉，
    #   可以将其改为 False 或去掉。
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        use_fast=False,
        trust_remote_code=True,
        local_files_only=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype="auto",
        device_map="auto",      # 在 NPU/Ascend 上，如果配套的 transformers / torch_npu
                                # 做了 device map 适配，可继续使用；否则可以改为 None 手动 .to("npu")
        local_files_only=True,
    )

    return model, tokenizer


def build_input(tokenizer, user_prompt: str, sys_prompt: str = None):
    """
    Build chat-style input using tokenizer's chat template.
    """
    messages = []

    # 如果你希望在对话模板中显式包含系统提示，可以将 sys_prompt 作为 system role 传入
    if sys_prompt:
        messages.append({"role": "system", "content": sys_prompt})

    messages.append({"role": "user", "content": user_prompt})

    # apply_chat_template 会根据 openPangu 的 chat 模板生成最终输入字符串
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt")
    return model_inputs


def generate(
    model,
    tokenizer,
    prompt: str,
    sys_prompt: str = None,
    max_new_tokens: int = 32768,
    temperature: float = 0.7,
    top_p: float = 0.9,
    do_sample: bool = True,
    eos_token_id: int = 45892,
):
    """
    Run text generation for a single prompt.
    """

    model_inputs = build_input(tokenizer, prompt, sys_prompt=sys_prompt)
    model_inputs = model_inputs.to(model.device)

    # 构建 GenerationConfig（可选，也可以直接放在 model.generate 的参数中）
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        eos_token_id=eos_token_id,
    )

    outputs = model.generate(
        **model_inputs,
        generation_config=gen_config,
        return_dict_in_generate=True,
    )

    input_length = model_inputs.input_ids.shape[1]
    generated_tokens = outputs.sequences[:, input_length:]
    content = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    return content


def main():
    parser = argparse.ArgumentParser(description="Inference script for fine-tuned openPangu-Embedded-1B")

    parser.add_argument(
        "--model_path",
        type=str,
        default="/opt/huawei/edu-apaas/src/init/luoyuping/pangu1B_training/outputs/float_test/bf16_false/final_model",
        help="Path to fine-tuned model directory or HF repo id.",
    )
    parser.add_argument(
        "--prompt",
        type=str,
        default="Explain the concept of neural networks in simple terms.",
        help="User input prompt.",
    )
    parser.add_argument(
        "--max_new_tokens",
        type=int,
        default=512,
        help="Maximum number of new tokens to generate.",
    )
    parser.add_argument(
        "--temperature",
        type=float,
        default=0.7,
        help="Sampling temperature.",
    )
    parser.add_argument(
        "--top_p",
        type=float,
        default=0.9,
        help="Nucleus sampling top_p.",
    )
    parser.add_argument(
        "--do_sample",
        action="store_true",
        help="Use sampling instead of greedy decoding.",
    )

    args = parser.parse_args()

    # 你自己的安全/合规 system prompt，可以根据需要修改或删减
    sys_prompt = (
        "你必须严格遵守法律法规和社会道德规范。"
        "生成任何内容时，都应避免涉及暴力、色情、恐怖主义、种族歧视、性别歧视等不当内容。"
        "一旦检测到输入或输出有此类倾向，应拒绝回答并发出警告。例如，如果输入内容包含暴力威胁或色情描述，"
        "应返回错误信息：“您的输入包含不当内容，无法处理。”"
    )

    model, tokenizer = load_model_and_tokenizer(args.model_path)

    output = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=args.prompt,
        sys_prompt=sys_prompt,
        max_new_tokens=args.max_new_tokens,
        temperature=args.temperature,
        top_p=args.top_p,
        do_sample=args.do_sample,
    )

    print("\n=== Model output ===\n")
    print(output)


if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [--model_path MODEL_PATH] [--prompt PROMPT]
                             [--max_new_tokens MAX_NEW_TOKENS]
                             [--temperature TEMPERATURE] [--top_p TOP_P]
                             [--do_sample]
ipykernel_launcher.py: error: unrecognized arguments: -f /home/service/.local/share/jupyter/runtime/kernel-d2be4f6f-d181-4464-8eb6-39b593154272.json


SystemExit: 2